# The final model as a decision tree
**The model** (`docs/findings.md` §5-§6): a `joint` encoder (class-balanced reconstruction + latent prediction + SIGReg) warmed up for 20k steps (**stage 1: availability**), then forked:
the all-frame reconstruction goes to 0 while the decoded prediction, the prediction-gated reconstruction (`tau` 0.005) and the target gradient come in over 4000 steps and stay for 8000 more
(**stage 2: judgement and consolidation**). On one seed it held every learned part, released the noise completely and kept the dynamics.

**What is still open, in the order that prunes the most runs.** Each stage tests one thing against the current defaults, a pre-registered rule makes the decision, and the decision becomes the default for every later stage.
Seeds and the 40k warmup come last, on the model that is left. Nothing is run twice: the ground-truth parent and its `final-w` fork exist already (notebook 08).

| stage | question | new runs | decision (written to `decisions.json`) |
|---|---|---|---|
| **0** | the reference: ground-truth weights, SIGReg 0.01, stage 2 with class weights | none (reused) | |
| **A** | does stage 2 need class weights? (the gate already selects tiles) | 1 fork | `weights2` |
| **B** | does the model need SIGReg? | 1 parent (`lam = 0`) + 1 fork | `lam` |
| **C** | what replaces the ground-truth weights in stage 1? `none` (plain MSE) and `focal` (error-weighted, label-free) | 2 parents + 2 forks | `stage1` |
| **T** | the **target gradient already in the parent**, from step 0 (reconstruction stays on) | 1 parent + 1 fork | `tgt_w` |
| **D** | **how much reconstruction is needed**: the reconstruction weight lowered to 0.3 and 0.1 (1 is the winner itself) | 2 parents + 2 forks | `rec_w` |
| **E** | seeds 1 and 2, and the 40k warmup, **for the final model only** | 3 parents + 3 forks | replication verdict |
| **F** | **the last experiment: limited capacity.** The final model at D = 64 and 32, **nothing else changed**: the parent and the final fork | 2 parents + 2 forks | verdict only |

At most **2 runs at a time** (the optimum on the A40). 11 new parents and 12 new forks in total, about 7 h end to end; a stage that fails shortens the rest.
Override any decision by hand in `OVERRIDE` (parameters cell).

**Design notes**
* *Error-weighted reconstruction* (`rec_focal = 1`): every tile's error is weighted by its own current error relative to the mean (stop-gradient, mean 1, cap 20): an L2 loss over the tiles. Small content that a plain MSE
  leaves unlearned is pulled in per sample, with no labels. It rewards irreducible noise too; in stage 1 that is intended (everything first becomes available), and in stage 2 the gate and the target gradient take it away.
* *The target gradient in the parent* (`tgt_w`): the prediction error also moves the real next latent (`||sg(P(z_ctx, a)) - z_next||^2`), next to the reconstruction, which stays on. Early on the predictor is bad, so this is the race (content it has not learned is pulled toward its wrong output), tested at its worst on purpose. With the reconstruction on I expect the noise to be shrunk but still readable (the decoder can scale it back up), so the fork's fade stays necessary; what it may add is better dynamics.
* *Stage 2 without weights* means plain pixel error in the decoded prediction and the gated reconstruction; the gate already selects tiles by predictability.
* *The reconstruction weight* is the knob the prediction loss is traded against: lowering it lets the unpredictable fall off first and shows how much reconstruction the content really needs. Stage 2 does the same in time (the reconstruction fades while the target gradient comes in).
* *One instrument*: a fresh decoder with fixed evaluation weights (power 1, cap 4) reads every run, so label-free models are compared with the same instrument; the labels are used only to read.
* Seeds vary the initialisation and the data stream, not the world.

**Pre-registered rules** (fresh decoder; a drop is reference minus new; `HOLD` = agent position, facing, hand, inventory; the mob is reported but never used, its readout is unreliable, `docs/findings.md` §7):
* **A:** drop weights in stage 2 if `final` is within 0.05 of `final-w` in HOLD, agent and terrain, within 0.1 in the predicted facing / hand / mob, and twinkle (fresh) <= 0.05.
* **B:** drop SIGReg if the `lam = 0` parent is within 0.05 of the reference parent in position, facing, hand, keeps rank >= 0.7 of it, **and** its fork is within 0.05 of the reference fork in HOLD with twinkle <= 0.05.
* **C:** a weighting replaces the labels if its parent is within 0.1 of the reference parent in position, facing, hand **and** its fork is within 0.1 of the reference fork in HOLD with twinkle <= 0.05. `none` is preferred over `focal` (simplest), else the labels stay (`gt`).
* **T:** the target gradient moves into the parent if the parent is within 0.05 of the winner's parent in HOLD, its fork is within 0.05 of the winner's fork in HOLD with twinkle <= 0.05, **and** the predicted hand or mob rises by >= 0.05 in the parent or the fork.
* **D:** the **lowest** reconstruction weight whose parent and fork both hold (within 0.05 of the weight-1 parent and fork in HOLD, fork twinkle <= 0.05) is adopted; if neither 0.3 nor 0.1 holds, 1 stays. The table shows the whole curve (HOLD, the predicted hand / mob, the twinkle in the parent), so the threshold is visible whichever way it falls.
* **F, capacity** (exploratory, fresh decoder; the curve over D = 128 / 64 / 32 is the result): *capacity binds* at a D if the parent loses >= 0.1 against the D = 128 parent in agent position, facing, hand, inventory or terrain. *Stage 2 still works* at a D if its fork is within 0.05 of its own parent in HOLD and its twinkle is <= 0.05.
* **E:** the final model replicates if in each seed the fork holds (drop <= 0.05 against its parent in HOLD), twinkle <= 0.05 and rank >= 0.7 of the parent's. The 40k parent is exploratory: it must hold, and the predicted hand or mob should be >= 0.05 above the 20k fork.

In [ ]:
%pip install -q -e ..
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU (CPU fallback, slow)")

In [ ]:
%matplotlib inline
import json
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from gaf.env_grid import GridCfg
from gaf.train import Cfg
from gaf import analysis as A
from gaf import readout as R
from gaf import fork as F

OUT = "/workspace/results/final_model"
REUSE_FROM = "/workspace/results/joint_dp"      # notebook 08's results: the reference parent and its final-w fork are copied from there instead of retrained
WORKERS = 2                                    # runs at a time (the optimum on an A40)
DIMS = (64, 32)                                # stage F: the latent sizes besides D = 128
OVERRIDE = {}                                  # force a decision, e.g. {"lam": 0.0, "weights2": False, "stage1": "focal", "rec_w": 0.3}
D, STEPS, STEPS_LONG = 128, 20_000, 40_000
FORK_STEPS, FADE, TAU = 12_000, 4_000, 0.005
DS_STEPS = 5000                                 # steps of the fresh decoder
EVAL_W = (1.0, 4.0)                             # evaluation class weights (power, cap): one instrument for every run
ENV = GridCfg(W=12, H=8, hand_tiles=1, agent_shape="arrow", mob_mode="chase", n_mobs=1, mob_chase=0.7)
COMMON = dict(lam=0.01, batch=256, lr=1e-3, wd=0.01, steps=STEPS, floor_start=10 ** 9, eval_every=2000, eval_parts=6000, eval_parts_steps=500,
              D=D, env=ENV, rec_class_power=1.0, rec_class_cap=4.0, pred_arch="film", ckpt_every=5000)
HOLD = ("fresh: agent position", "fresh: facing", "fresh: hand", "fresh: inventory")
KEY = ["fresh: agent position", "fresh: facing", "fresh: agent", "fresh: hand", "fresh: inventory", "fresh: terrain", "fresh: mob",
       "twinkle: fresh", "twinkle: own decoder", "pred: facing (move)", "pred: hand", "pred: mob", "latent rank"]
pd.set_option("display.max_columns", None, "display.width", 250, "display.precision", 2)

# ---- decisions: each stage writes one, every later stage reads them
DEFAULTS = {"weights2": True, "lam": 0.01, "stage1": "gt", "tgt_w": 0.0, "rec_w": 1.0}
DEC_FILE = Path(OUT) / "decisions.json"


def decisions():
    saved = json.loads(DEC_FILE.read_text()) if DEC_FILE.exists() else {}
    return {**DEFAULTS, **{k: v for k, v in saved.items() if k in DEFAULTS}, **OVERRIDE}


def decide(key, value, why):
    DEC_FILE.parent.mkdir(parents=True, exist_ok=True)
    saved = json.loads(DEC_FILE.read_text()) if DEC_FILE.exists() else {}
    saved[key], saved[key + " (why)"] = value, why
    DEC_FILE.write_text(json.dumps(saved, indent=1))
    print(f"DECISION {key} = {value}   ({why})")


# ---- runs: a parent is a name -> Cfg, a fork is (parent name, recipe)
PARENTS, PDIR, FORKS = {}, {}, []
RECIPES = {"final-w": dict(r_floor=0.0, dp=1.0, tgt=1.0, gate=TAU, weights=True),     # class weights in stage 2 (the earlier candidate)
           "final":   dict(r_floor=0.0, dp=1.0, tgt=1.0, gate=TAU, weights=False)}    # no labels in stage 2
tag_of = lambda recipe: recipe.replace("-", "")
final_recipe = lambda: "final-w" if decisions()["weights2"] else "final"


def spec(kind, lam, tgt_w=0.0, rec_w=1.0):
    """The name and the tag of a parent, deterministic from its settings (so a restarted kernel rebuilds the same run ids)."""
    tag = {"gt": "dp-joint" if lam > 0 else "gt-nosig", "none": "u", "focal": "f"}[kind]
    name = {"gt": "gt s0" if lam > 0 else "gt nosig", "none": "none", "focal": "focal"}[kind]
    if tgt_w > 0:
        tag, name = tag + "-tgton", name + " tgt on"
    if rec_w != 1.0:
        tag, name = tag + f"-r{rec_w:g}", name + f" rec{rec_w:g}"
    return name, tag


def add_parent(kind, lam, tgt_w=0.0, rec_w=1.0, seed=0, steps=STEPS, dim=None):
    name, tag = spec(kind, lam, tgt_w, rec_w)
    if dim is not None and dim != D:
        name, tag = name + f" D{dim}", tag + f"-D{dim}"
    if seed:
        name += f" s{seed}"
    if steps != STEPS:
        name, tag = name + f" {steps / 1000:g}k", tag + f"-{steps / 1000:g}k"
    kw = {"gt": dict(), "none": dict(rec_class_power=0.0), "focal": dict(rec_class_power=0.0, rec_focal=1.0)}[kind]
    PARENTS[name] = Cfg(arm="joint", tag=tag, seed=seed, **{**COMMON, "D": D if dim is None else dim, "steps": steps, "lam": lam, "shadow_rec": rec_w, "tgt_w": tgt_w, **kw})
    PDIR[name] = Path(OUT) / PARENTS[name].run_id
    return name


def winner(**over):
    """The model the decisions so far describe (stage 1 weighting, SIGReg, target gradient, reconstruction weight)."""
    d = decisions()
    return {**dict(kind=d["stage1"], lam=d["lam"], tgt_w=d["tgt_w"], rec_w=d["rec_w"]), **over}


winner_name = lambda **over: spec(**winner(**over))[0]


def add_fork(parent, recipe):
    if (parent, recipe) not in FORKS:
        FORKS.append((parent, recipe))


fdir = lambda parent, recipe: PDIR[parent] / "forks" / F.fork_name("fade", tag_of(recipe))
parent_done = lambda n: (PDIR[n] / "summary.json").exists()
fork_done = lambda p, r: (fdir(p, r) / "done").exists()
fork_running = lambda p, r: (fdir(p, r) / "pid").exists() and not fork_done(p, r) and F._alive(fdir(p, r) / "pid")


def launch_parents(names, workers=WORKERS):
    A.launch([PARENTS[n] for n in names], OUT, workers=workers)


def launch_forks(pairs=None, limit=WORKERS):
    """Start the forks whose parent is done, keeping at most `limit` forks running."""
    pairs = FORKS if pairs is None else pairs
    n_run = sum(fork_running(p, r) for p, r in pairs)
    for p, r in pairs:
        if n_run >= limit:
            break
        if parent_done(p) and not fork_done(p, r) and not fork_running(p, r):
            kw = dict(RECIPES[r])
            r_floor = kw.pop("r_floor")
            F.launch_forks(PDIR[p], modes=("fade",), steps=FORK_STEPS, fade=FADE, r_floor=r_floor, tag=tag_of(r), **kw)
            n_run += 1
            print("started fork", p, "|", r)


def status(parents=(), pairs=()):
    rows = {}
    for n in parents:
        pr = A.progress(OUT, [PARENTS[n]]).iloc[0]
        rows[n] = {"state": pr["state"], "step": pr["step"], "of": pr["of"]}
    for p, r in pairs:
        h = pd.read_csv(fdir(p, r) / "metrics.csv") if (fdir(p, r) / "metrics.csv").exists() else None
        rows[f"{p} | {r}"] = {"state": "done" if fork_done(p, r) else ("running" if fork_running(p, r) else ("waiting for parent" if not parent_done(p) else "not started")),
                              "step": int(h.step.iloc[-1]) if h is not None else 0, "of": FORK_STEPS}
    display(pd.DataFrame(rows).T)


# ---- readouts: computed once per run, cached in its folder; display never trains
def eff_rank(d):  # participation ratio of the live latent's covariance spectrum
    ck = R.load(d)
    with torch.no_grad():
        z = R.systems(ck)["live"][0](ck.env.sample(2048).x[:, -1])
    ev = torch.linalg.eigvalsh(torch.cov(z.T))
    return float(ev.sum() ** 2 / ev.pow(2).sum())


def readouts(d, force=False):
    cache = Path(d) / "readouts_final.json"
    if cache.exists() and not force:
        return json.loads(cache.read_text())
    dyn, con = R.dynamics(d).loc["live"], R.decoder_strength(d, steps=DS_STEPS, weights=EVAL_W).loc["live"]
    out = {"fresh: agent position": con["agent position acc"], "fresh: facing": con["facing acc"], "fresh: agent": con["agent"], "fresh: hand": con["hand"],
           "fresh: inventory": con["inventory"], "fresh: terrain": con["map"], "fresh: mob": con["mob"], "twinkle: fresh": con["twinkle"],
           "twinkle: own decoder": dyn.loc["actual next frame", "twinkle"],
           **{f"pred: {k}": dyn.loc["predicted", c] for k, c in (("facing (move)", "facing acc, move steps"), ("hand", "hand"), ("mob", "mob"), ("agent", "agent"))},
           "latent rank": eff_rank(d)}
    out = {k: float(v) for k, v in out.items()}
    cache.write_text(json.dumps(out))
    return out


def table(parents=(), pairs=(), save=None):
    """Readouts of the finished runs among the given parents and forks (the fresh decoder trains once for each, then it is cached)."""
    rows = {}
    for n in parents:
        if parent_done(n):
            print(n, "cached" if (PDIR[n] / "readouts_final.json").exists() else "computing ...", flush=True)
            rows[n] = readouts(PDIR[n])
    for p, r in pairs:
        if fork_done(p, r):
            print(f"{p} | {r}", "cached" if (fdir(p, r) / "readouts_final.json").exists() else "computing ...", flush=True)
            rows[f"{p} | {r}"] = readouts(fdir(p, r))
    t = pd.DataFrame(rows).T
    if save:
        t.to_csv(Path(OUT) / f"{save}.csv")
    display(t[KEY].round(2))
    return t


def diff(t, ref, new, keys):
    """reference minus new, for the given readouts (positive = the new run is worse)."""
    return {k.split(": ")[-1]: round(float(t.loc[ref, k] - t.loc[new, k]), 2) for k in keys}


def within(d, tol):
    return all(v <= tol for v in d.values())
print("ready;", "decisions:", decisions())

## Stage 0: the reference (reused)
The ground-truth parent (seed 0, SIGReg 0.01, class weights cap 4) and its `final-w` fork exist in notebook 08's results (the fork is `fade-dpr0gatetgt005`: decoded prediction, gate tau 0.005 and target gradient, with class weights).
They are copied; if `REUSE_FROM` does not have them they train here (about 50 min).

In [ ]:
add_parent("gt", 0.01)                      # name "gt s0"
add_fork("gt s0", "final-w")
src_p = Path(REUSE_FROM) / PARENTS["gt s0"].run_id
if src_p.exists() and (src_p / "summary.json").exists() and not PDIR["gt s0"].exists():
    skip = shutil.ignore_patterns("forks", "readouts*", "fresh_decoder.csv", "gate_*", "action_test.csv", "agent_split.csv", "mob_position.csv")
    shutil.copytree(src_p, PDIR["gt s0"], ignore=skip)
    print("reused the parent", PARENTS["gt s0"].run_id)
src_f = src_p / "forks" / "fade-dpr0gatetgt005"
if src_f.exists() and (src_f / "done").exists() and not fdir("gt s0", "final-w").exists():
    shutil.copytree(src_f, fdir("gt s0", "final-w"), ignore=shutil.ignore_patterns("readouts*", "pid"))
    print("reused the fork final-w")
launch_parents(["gt s0"])        # nothing to start when it was reused
launch_forks()

## Stages A and B (first half), together: two runs at a time
**A** is a fork only: stage 2 without class weights, from the reference parent. **B** needs a parent without SIGReg (`lam = 0`, 20k steps), which runs next to it; its fork waits for the decision of A (stage B, second half).

In [ ]:
add_fork("gt s0", "final")                               # A: stage 2 without class weights
add_parent("gt", 0.0)                                    # B: the parent without SIGReg (name "gt nosig")
launch_forks(pairs=[("gt s0", "final")], limit=1)
launch_parents(["gt nosig"], workers=1)

In [ ]:
status(parents=["gt s0", "gt nosig"], pairs=[("gt s0", "final-w"), ("gt s0", "final")])

In [ ]:
# decision A
tA = table(pairs=[("gt s0", "final-w"), ("gt s0", "final")], parents=["gt s0"], save="stage_A")
a, b = "gt s0 | final-w", "gt s0 | final"
if a in tA.index and b in tA.index:
    d1 = diff(tA, a, b, ("fresh: agent position", "fresh: facing", "fresh: agent", "fresh: hand", "fresh: inventory", "fresh: terrain"))
    d2 = diff(tA, a, b, ("pred: facing (move)", "pred: hand", "pred: mob"))
    ok = within({k: abs(v) for k, v in d1.items()}, 0.05) and within({k: abs(v) for k, v in d2.items()}, 0.1) and tA.loc[b, "twinkle: fresh"] <= 0.05
    decide("weights2", not ok, f"final against final-w: content {d1}, dynamics {d2}, twinkle {tA.loc[b, 'twinkle: fresh']:.2f} -> " + ("weights off in stage 2" if ok else "keep class weights in stage 2"))
else:
    print("wait: the fork `final` is not done yet")

## Stage B, second half: the fork of the parent without SIGReg
It uses the recipe chosen in A. Its reference is the ground-truth parent with SIGReg and the same recipe.

In [ ]:
add_fork("gt nosig", final_recipe())
add_fork("gt s0", final_recipe())
launch_forks(pairs=[("gt nosig", final_recipe())])
status(parents=["gt nosig"], pairs=[("gt nosig", final_recipe())])

In [ ]:
# decision B
fr = final_recipe()
tB = table(parents=["gt s0", "gt nosig"], pairs=[("gt s0", fr), ("gt nosig", fr)], save="stage_B")
if "gt nosig" in tB.index and f"gt nosig | {fr}" in tB.index:
    dp_ = diff(tB, "gt s0", "gt nosig", ("fresh: agent position", "fresh: facing", "fresh: hand"))
    df_ = diff(tB, f"gt s0 | {fr}", f"gt nosig | {fr}", HOLD)
    rank_ok = tB.loc["gt nosig", "latent rank"] >= 0.7 * tB.loc["gt s0", "latent rank"]
    ok = within(dp_, 0.05) and rank_ok and within(df_, 0.05) and tB.loc[f"gt nosig | {fr}", "twinkle: fresh"] <= 0.05
    decide("lam", 0.0 if ok else 0.01, f"parent drops {dp_}, rank {tB.loc['gt nosig', 'latent rank']:.0f} vs {tB.loc['gt s0', 'latent rank']:.0f}, fork drops {df_} -> " + ("SIGReg not needed" if ok else "keep SIGReg"))
else:
    print("wait: the parent or its fork is not done yet")

## Stage C: what replaces the ground-truth weights in stage 1
Two new parents with the decisions so far (SIGReg as decided): **`none`** (plain pixel MSE) and **`focal`** (error-weighted, label-free), then the fork recipe chosen in A on each.
The reference is the ground-truth parent with the same SIGReg setting.

In [ ]:
ref_parent = spec("gt", decisions()["lam"])[0]
add_parent("none", decisions()["lam"])
add_parent("focal", decisions()["lam"])
for n in ("none", "focal"):
    add_fork(n, final_recipe())
add_fork(ref_parent, final_recipe())
launch_parents(["none", "focal"])

In [ ]:
launch_forks(pairs=[("none", final_recipe()), ("focal", final_recipe())])     # re-run while the parents finish
status(parents=["none", "focal"], pairs=[("none", final_recipe()), ("focal", final_recipe())])

In [ ]:
# decision C
fr = final_recipe()
ref_parent = spec("gt", decisions()["lam"])[0]
tC = table(parents=[ref_parent, "none", "focal"], pairs=[(ref_parent, fr), ("none", fr), ("focal", fr)], save="stage_C")
passed, notes = {}, {}
for n in ("none", "focal"):
    if n in tC.index and f"{n} | {fr}" in tC.index:
        dp_ = diff(tC, ref_parent, n, ("fresh: agent position", "fresh: facing", "fresh: hand"))
        df_ = diff(tC, f"{ref_parent} | {fr}", f"{n} | {fr}", HOLD)
        passed[n] = within(dp_, 0.1) and within(df_, 0.1) and tC.loc[f"{n} | {fr}", "twinkle: fresh"] <= 0.05
        notes[n] = f"{n}: parent drops {dp_}, fork drops {df_}, twinkle {tC.loc[f'{n} | {fr}', 'twinkle: fresh']:.2f} -> {'passes' if passed[n] else 'fails'}"
if len(passed) == 2:
    chosen = "none" if passed["none"] else ("focal" if passed["focal"] else "gt")
    decide("stage1", chosen, "; ".join(notes.values()) + (" -> the labels stay" if chosen == "gt" else f" -> {chosen} replaces the labels"))
else:
    print("wait: parents or forks not done yet", list(passed))

## Stage T: the target gradient already in the parent
The stage 1 winner with the target gradient **on from step 0** (weight 1), next to the reconstruction, which stays. The fork gets the recipe from A; the reference is the winner without it.

In [ ]:
base = winner_name(tgt_w=0.0, rec_w=1.0)
T_NAME = add_parent(**winner(tgt_w=1.0, rec_w=1.0))
for n in (T_NAME, base):
    add_fork(n, final_recipe())
launch_parents([T_NAME])

In [ ]:
launch_forks(pairs=[(T_NAME, final_recipe())])     # re-run while the parent trains
status(parents=[T_NAME], pairs=[(T_NAME, final_recipe())])

In [ ]:
# decision T
fr = final_recipe()
base = winner_name(tgt_w=0.0, rec_w=1.0)
tT = table(parents=[base, T_NAME], pairs=[(n, fr) for n in (base, T_NAME)], save="stage_T")
if T_NAME in tT.index and f"{T_NAME} | {fr}" in tT.index and f"{base} | {fr}" in tT.index:
    dpar = diff(tT, base, T_NAME, HOLD)
    dfork = diff(tT, f"{base} | {fr}", f"{T_NAME} | {fr}", HOLD)
    gain = max([tT.loc[T_NAME, f"pred: {k}"] - tT.loc[base, f"pred: {k}"] for k in ("hand", "mob")]
               + [tT.loc[f"{T_NAME} | {fr}", f"pred: {k}"] - tT.loc[f"{base} | {fr}", f"pred: {k}"] for k in ("hand", "mob")])
    ok = bool(within(dpar, 0.05) and within(dfork, 0.05) and tT.loc[f"{T_NAME} | {fr}", "twinkle: fresh"] <= 0.05 and gain >= 0.05)
    decide("tgt_w", 1.0 if ok else 0.0, f"parent drops {dpar}, fork drops {dfork}, predicted gain {gain:.2f}, twinkle {tT.loc[f'{T_NAME} | {fr}', 'twinkle: fresh']:.2f} -> " + ("target gradient in the parent" if ok else "the target gradient stays out of the parent"))
else:
    print("wait: the parent or its fork is not done yet")

## Stage D: how much reconstruction is needed
On the stage 1 winner: the reconstruction weight lowered to 0.3 and 0.1 (1 is the winner itself); the prediction loss keeps weight 1. Each parent gets the fork recipe from A. The question is the threshold: the lowest weight at which the content is still held.

In [ ]:
base = winner_name(rec_w=1.0)
for w in (0.3, 0.1):
    add_parent(**winner(rec_w=w))
    add_fork(winner_name(rec_w=w), final_recipe())
add_fork(base, final_recipe())
D_NAMES = [winner_name(rec_w=0.3), winner_name(rec_w=0.1)]
launch_parents(D_NAMES)

In [ ]:
D_NAMES = [winner_name(rec_w=0.3), winner_name(rec_w=0.1)]
launch_forks(pairs=[(n, final_recipe()) for n in D_NAMES])
status(parents=D_NAMES, pairs=[(n, final_recipe()) for n in D_NAMES])

In [ ]:
# decision D
fr = final_recipe()
names = {1.0: winner_name(rec_w=1.0), 0.3: winner_name(rec_w=0.3), 0.1: winner_name(rec_w=0.1)}
base = names[1.0]
tD = table(parents=list(names.values()), pairs=[(n, fr) for n in names.values()], save="stage_D")
holds = {}
for w in (0.3, 0.1):
    n = names[w]
    if n in tD.index and f"{n} | {fr}" in tD.index and base in tD.index and f"{base} | {fr}" in tD.index:
        dpar = diff(tD, base, n, HOLD)
        dfork = diff(tD, f"{base} | {fr}", f"{n} | {fr}", HOLD)
        holds[w] = (bool(within(dpar, 0.05) and within(dfork, 0.05) and tD.loc[f"{n} | {fr}", "twinkle: fresh"] <= 0.05), dpar, dfork)
if len(holds) == 2:
    ok = [w for w, h in holds.items() if h[0]]
    best = min(ok) if ok else 1.0
    decide("rec_w", best, f"{holds} -> " + (f"the lowest weight that holds is {best}" if ok else "neither 0.3 nor 0.1 holds, keep 1.0"))
else:
    print("wait: parents or forks not done yet")

## Stage E: seeds and the 40k warmup, for the final model only
The configuration is now fixed by the decisions. Seed 0 is the model that won the stages; **seeds 1 and 2** and the **40k parent** (seed 0) are new. Each parent gets the final recipe.

In [ ]:
fr = final_recipe()
w0 = winner_name()                                   # seed 0: the model that won the stages
FINAL_NAMES = [w0] + [add_parent(**winner(), seed=s) for s in (1, 2)] + [add_parent(**winner(), steps=STEPS_LONG)]
for n in FINAL_NAMES:
    add_fork(n, fr)
print("final model:", winner(), "| recipe", fr, "| seed 0 is", w0)
launch_parents(FINAL_NAMES[1:])

In [ ]:
launch_forks(pairs=[(n, final_recipe()) for n in FINAL_NAMES])
status(parents=FINAL_NAMES[1:], pairs=[(n, final_recipe()) for n in FINAL_NAMES])

In [ ]:
# verdict E
fr = final_recipe()
w0 = FINAL_NAMES[0]
tE = table(parents=FINAL_NAMES, pairs=[(n, fr) for n in FINAL_NAMES], save="stage_E")
lines = []
emit = lambda *a: (print(*a), lines.append(" ".join(str(x) for x in a)))
emit("E replication of the final model:", decisions())
for n in FINAL_NAMES[:3]:
    f = f"{n} | {fr}"
    if f in tE.index:
        d = diff(tE, n, f, HOLD)
        ok = within(d, 0.05) and tE.loc[f, "twinkle: fresh"] <= 0.05 and tE.loc[f, "latent rank"] >= 0.7 * tE.loc[n, "latent rank"]
        emit(f"  {n}: {'yes' if ok else 'NO'}  drops {d}, twinkle {tE.loc[f, 'twinkle: fresh']:.2f}, rank {tE.loc[f, 'latent rank']:.0f} vs {tE.loc[n, 'latent rank']:.0f}")
f40, f20 = f"{FINAL_NAMES[3]} | {fr}", f"{FINAL_NAMES[0]} | {fr}"
if f40 in tE.index and f20 in tE.index:
    d = diff(tE, FINAL_NAMES[0], f40, HOLD)
    up = {k: round(float(tE.loc[f40, f"pred: {k}"] - tE.loc[f20, f"pred: {k}"]), 2) for k in ("hand", "mob")}
    emit(f"  40k warmup (exploratory): holds {'yes' if within(d, 0.05) else 'NO'} {d}; predicted gain over the 20k fork {up}")
(Path(OUT) / "verdict_E.txt").write_text("\n".join(lines))

## Stage F, the last experiment: limited capacity
The final model (the decisions so far) at **D = 64 and D = 32**; D = 128 is the winner itself. **Nothing changes except D**: for each D the parent and the final fork, the same recipe and the same decisions. The question is what the model holds
and releases when the latent is smaller than what the world needs. Run the launch cell; the fork cell when the parents are done (so that no more than two runs compete).

In [ ]:
w0 = winner_name()
F_PARENTS = {128: w0}
for dim in DIMS:
    F_PARENTS[dim] = add_parent(**winner(), dim=dim)
F_PAIRS = [(F_PARENTS[dim], final_recipe()) for dim in DIMS]
for p_, r_ in F_PAIRS:
    add_fork(p_, r_)
launch_parents([F_PARENTS[d] for d in DIMS])

In [ ]:
launch_forks(pairs=F_PAIRS)     # re-run while the parents train
status(parents=[F_PARENTS[d] for d in DIMS], pairs=F_PAIRS)

In [ ]:
# verdict F
fr = final_recipe()
pairs = [(n, fr) for n in F_PARENTS.values()]
tF = table(parents=list(F_PARENTS.values()), pairs=pairs, save="stage_F")
ITEMS = HOLD + ("fresh: terrain",)
lines = []
emit = lambda *a: (print(*a), lines.append(" ".join(str(x) for x in a)))
ref = F_PARENTS[128]
emit("F capacity: the final model at D = 128 / 64 / 32")
for dim in (128,) + tuple(DIMS):
    n = F_PARENTS[dim]
    f = f"{n} | {fr}"
    if n not in tF.index:
        continue
    drop = diff(tF, ref, n, ITEMS) if dim != 128 else {}
    emit(f"  D={dim}: parent: capacity binds: {'yes' if any(v >= 0.1 for v in drop.values()) else 'no'} {drop}; rank {tF.loc[n, 'latent rank']:.0f}")
    if f in tF.index:
        d = diff(tF, n, f, HOLD)
        ok = within(d, 0.05) and tF.loc[f, "twinkle: fresh"] <= 0.05
        emit(f"       fork: stage 2 still works: {'yes' if ok else 'NO'}  drops against its parent {d}, twinkle {tF.loc[f, 'twinkle: fresh']:.2f}, rank {tF.loc[f, 'latent rank']:.0f}")
(Path(OUT) / "verdict_F.txt").write_text("\n".join(lines))

## Summary
Everything is in `<OUT>/`: `decisions.json` (every decision with its reason), `stage_A.csv` ... `stage_F.csv`, `verdict_E.txt`, `verdict_F.txt`, and a `readouts_final.json` in every run folder. The checkpoints allow any readout to be redone without training.
Zip `/workspace/results/final_model` (checkpoints and `forks/` included) and download it.

## Which cells to run
Setup (once, and again after a restart); then per stage: the launch cell, the status cell (re-run while waiting; the fork launch cells are safe to re-run), the decision cell. Stages A and B run together; C, T, D, E, F in order.